# 1. Preprocessing и Split Strategy для Movie Recommender

Здесь будем готовить данные для следующего этапа моделирования рекомендательной системы

- EDA показал сильную разреженность, long-tail и необходимость chronological split
- Для baseline-пайплайна фиксируем implicit-постановку: **positive = rating >= 4**

Что делает ноутбук:
- загружает активный датасет и конфиг
- готовит interactions для modeling
- строит chronological split без leakage
- формирует warm-start валидацию/тест
- сохраняет артефакты для popularity, item-item CF и matrix factorization


In [1]:
# 2. Импорты и настройки
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 140)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

project_root = Path.cwd()
if not ((project_root / 'configs').exists() or (project_root / 'config').exists()):
    parent = project_root.parent
    if (parent / 'configs').exists() or (parent / 'config').exists():
        project_root = parent

CONFIG_DIR = project_root / 'configs'
if not CONFIG_DIR.exists():
    CONFIG_DIR = project_root / 'config'

DATA_DIR = project_root / 'data'
CONFIG_PATH = CONFIG_DIR / 'base.json'
artifacts_root_dir = 'artifacts'
if CONFIG_PATH.exists():
    with CONFIG_PATH.open('r', encoding='utf-8') as f:
        _cfg = json.load(f)
    artifacts_root_dir = (
        _cfg.get('paths', {}).get('artifacts_root_dir')
        or _cfg.get('data', {}).get('artifacts_root_dir')
        or 'artifacts'
    )
ARTIFACTS_DIR = project_root / artifacts_root_dir
SPLITS_DIR = ARTIFACTS_DIR / 'splits'
PROCESSED_DIR = ARTIFACTS_DIR / 'processed'
EVAL_DIR = SPLITS_DIR / 'eval'

for p in [ARTIFACTS_DIR, SPLITS_DIR, PROCESSED_DIR, EVAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('project_root =', project_root)
print('data_dir =', DATA_DIR)
print('artifacts_dir =', ARTIFACTS_DIR)
print('random_seed =', RANDOM_SEED)


project_root = /Users/maybeabakarov/Desktop/РекомендацииПроект
data_dir = /Users/maybeabakarov/Desktop/РекомендацииПроект/data
artifacts_dir = /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts_new
random_seed = 42


## 3. Загрузка данных и конфига

Читаем конфиг активного датасета и загружаем таблицы на его основе

Критичные поля для пайплайна:
- `ratings`: `userId`, `movieId`, `rating`, `timestamp`

Вспомогательные поля:
- `movies`: metadata для будущего hybrid
- `tags` и `links`: дополнительное enrichment


In [2]:
config_path = CONFIG_DIR / 'base.json'

with config_path.open('r', encoding='utf-8') as f:
    config = json.load(f)

data_cfg = config['data']
active_dataset = data_cfg['active_dataset']
datasets_cfg = data_cfg['datasets']

raw_root = Path(data_cfg.get('raw_root_dir', 'data/raw'))
if not raw_root.is_absolute():
    raw_root = project_root / raw_root

ds_cfg = datasets_cfg[active_dataset]
dataset_dir = raw_root / ds_cfg.get('raw_subdir', active_dataset)

file_map = {
    'ratings.csv': dataset_dir / ds_cfg.get('ratings_filename', 'ratings.csv'),
    'movies.csv': dataset_dir / ds_cfg.get('movies_filename', 'movies.csv'),
    'tags.csv': dataset_dir / ds_cfg.get('tags_filename', 'tags.csv'),
    'links.csv': dataset_dir / ds_cfg.get('links_filename', 'links.csv'),
}

print('active_dataset:', active_dataset)
print('dataset_dir:', dataset_dir)

tables = {}
for fname, fpath in file_map.items():
    df = pd.read_csv(fpath)
    if 'timestamp' in df.columns:
        if pd.api.types.is_numeric_dtype(df['timestamp']):
            df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', errors='coerce')
        else:
            df['timestamp_dt'] = pd.to_datetime(df['timestamp'], errors='coerce')
    tables[fname.replace('.csv', '')] = df

ratings = tables.get('ratings')
movies = tables.get('movies')
tags = tables.get('tags')
links = tables.get('links')


active_dataset: ml-latest
dataset_dir: /Users/maybeabakarov/Desktop/РекомендацииПроект/data/raw/ml-latest


## 4. Базовая подготовка interactions

In [3]:
required_cols = {'userId', 'movieId', 'rating', 'timestamp'}
interactions = ratings[['userId', 'movieId', 'rating', 'timestamp']].copy()

# приведение типов
interactions['timestamp'] = interactions['timestamp'].astype('int64')
interactions['timestamp_dt'] = pd.to_datetime(interactions['timestamp'], unit='s', errors='coerce')
interactions = interactions[interactions['timestamp_dt'].notna()].copy()

display(interactions['rating'].value_counts().sort_index().to_frame('count'))



,count
rating,
0.5,566306
1.0,1013645
1.5,562409
2.0,2146492
2.5,1760733
3.0,6400664
3.5,4465001
4.0,8835955
4.5,3123055


### Что это значит для рекомендательной системы

- На этом шаге формируется стабильная interaction-таблица для reproducible split
- Временная сортировка и проверка типов — обязательная база для корректного chronological pipeline
- Мы сохраняем максимально «сырые», но валидные interactions без агрессивной фильтрации


## 6. Правила фильтрации перед split

In [4]:
# Основная постановка для последующих baseline-ноутбуков:
# implicit recommendation, positive interaction = rating >= 4.0
### Что это значит для рекомендательной системы

#- Для top-K рекомендаций естественна implicit-постановка: мы предсказываем вероятность релевантного взаимодействия.
#- Для baseline pipeline фиксируем **positive = rating >= 4**.
#- Explicit-версия сохраняется как reference, но основной workflow для следующих шагов — implicit.
POSITIVE_THRESHOLD = 4.0
implicit_positive = interactions[interactions['rating'] >= POSITIVE_THRESHOLD].copy()

raw_implicit = implicit_positive[['userId', 'movieId', 'rating', 'timestamp', 'timestamp_dt']].copy()

# Базовая фильтрация: минимальный кол-во взаимодейстивий для устойчивых baseline-оценок
MIN_USER_POS = 2
MIN_ITEM_POS = 2

# - 'cap': ограничиваем вклад каждого пользователя
# - 'drop': удаляем пользователей выше порога
# - 'none': оставляем как есть
USER_ACTIVITY_CONTROL_MODE = 'none'  # 'cap' | 'drop' | 'none'
CAP_USER_INTERACTIONS = 1000
DROP_USER_INTERACTIONS_THRESHOLD = 1000

user_pos_counts = raw_implicit.groupby('userId').size()
item_pos_counts = raw_implicit.groupby('movieId').size()

filtered_implicit = raw_implicit[
    raw_implicit['userId'].isin(user_pos_counts[user_pos_counts >= MIN_USER_POS].index)
].copy()
filtered_implicit = filtered_implicit[
    filtered_implicit['movieId'].isin(item_pos_counts[item_pos_counts >= MIN_ITEM_POS].index)
].copy()


def cap_user_interactions(df: pd.DataFrame, cap_value: int) -> pd.DataFrame:
    if cap_value is None or cap_value <= 0:
        return df.copy()
    tmp = df.sort_values(['userId', 'timestamp'], ascending=[True, False]).copy()
    tmp = tmp.groupby('userId', group_keys=False).head(cap_value)
    return tmp.sort_values('timestamp').reset_index(drop=True)


# Вариант 1: cap
capped_implicit = cap_user_interactions(filtered_implicit, CAP_USER_INTERACTIONS)

# Вариант 2: drop heavy users
user_counts_filtered = filtered_implicit.groupby('userId').size()
heavy_users_to_drop = set(user_counts_filtered[user_counts_filtered > DROP_USER_INTERACTIONS_THRESHOLD].index)
dropped_implicit = filtered_implicit[~filtered_implicit['userId'].isin(heavy_users_to_drop)].copy()

# Диагностика активности топ-пользователей
heavy_users_before = filtered_implicit.groupby('userId').size().sort_values(ascending=False).head(10)
heavy_users_after_cap = capped_implicit.groupby('userId').size().sort_values(ascending=False).head(10)
heavy_users_after_drop = dropped_implicit.groupby('userId').size().sort_values(ascending=False).head(10)

activity_diag = pd.DataFrame({
    'top_userId_before': heavy_users_before.index.astype('int64'),
    'n_interactions_before': heavy_users_before.values,
    'top_userId_after_cap': heavy_users_after_cap.index.astype('int64'),
    'n_interactions_after_cap': heavy_users_after_cap.values,
    'top_userId_after_drop': heavy_users_after_drop.index.astype('int64'),
    'n_interactions_after_drop': heavy_users_after_drop.values,
})

display(activity_diag)

# Выбор рабочего варианта
if USER_ACTIVITY_CONTROL_MODE == 'cap':
    main_implicit = capped_implicit.copy()
    main_variant_name = f'capped_implicit(cap={CAP_USER_INTERACTIONS})'
elif USER_ACTIVITY_CONTROL_MODE == 'drop':
    main_implicit = dropped_implicit.copy()
    main_variant_name = f'dropped_implicit(drop_users>{DROP_USER_INTERACTIONS_THRESHOLD})'
else:
    main_implicit = filtered_implicit.copy()
    main_variant_name = f'filtered_implicit(u>={MIN_USER_POS}, i>={MIN_ITEM_POS})'

print(f'Выбран основной вариант для baseline pipeline: {main_variant_name}')


,top_userId_before,n_interactions_before,top_userId_after_cap,n_interactions_after_cap,top_userId_after_drop,n_interactions_after_drop
0,189614,4439,189614,1000,170180,1000
1,44970,4049,122030,1000,86483,1000
2,196817,3829,124659,1000,227379,997
3,327439,3252,279508,1000,193799,995
4,33457,3213,211422,1000,112679,994
5,230765,3121,124544,1000,197138,994
6,248317,2649,327439,1000,40334,991
7,154203,2600,124510,1000,106257,990
8,256904,2543,279022,1000,316256,989
9,236260,2494,25611,1000,313408,988


Выбран основной вариант для baseline pipeline: filtered_implicit(u>=2, i>=2)


- Теперь есть три режима контроля гиперактивных пользователей: `cap`, `drop`, `none`
- `cap` ограничивает вклад, но сохраняет пользователя в данных
- `drop` удаляет пользователей выше порога активности
- Переключение делается одним параметром `USER_ACTIVITY_CONTROL_MODE`, что удобно для честного A/B сравнения метрик


## 7. Time split без leakage

In [5]:
# Глобальные временные квантильные границы
q80 = int(main_implicit['timestamp'].quantile(0.8))
q90 = int(main_implicit['timestamp'].quantile(0.9))

split_df = main_implicit.copy()
split_df['split'] = np.where(
    split_df['timestamp'] <= q80,
    'train',
    np.where(split_df['timestamp'] <= q90, 'validation', 'test')
)

train_implicit = split_df[split_df['split'] == 'train'].copy()
val_implicit = split_df[split_df['split'] == 'validation'].copy()
test_implicit = split_df[split_df['split'] == 'test'].copy()

split_summary = pd.DataFrame([
    {
        'split': name,
        'interactions': len(df),
        'users': df['userId'].nunique(),
        'items': df['movieId'].nunique(),
        'time_min': df['timestamp_dt'].min(),
        'time_max': df['timestamp_dt'].max(),
    }
    for name, df in [('train', train_implicit), ('validation', val_implicit), ('test', test_implicit)]
])
display(split_summary)


print('q80 cutoff:', pd.to_datetime(q80, unit='s'))
print('q90 cutoff:', pd.to_datetime(q90, unit='s'))



,split,interactions,users,items,time_min,time_max
0,train,13509884,264422,29796,1995-01-09 11:46:44,2018-07-11 02:45:48
1,validation,1688735,30505,28762,2018-07-11 02:46:00,2020-08-22 16:40:07
2,test,1688736,30840,28931,2020-08-22 16:40:53,2023-07-20 08:53:33


q80 cutoff: 2018-07-11 02:45:50
q90 cutoff: 2020-08-22 16:40:34


### Что это значит для рекомендательной системы

- Chronological split предотвращает утечку будущей информации в train
- Такой split ближе к реальному сценарию: модель обучается на прошлом и оценивается на будущем
- Ограничение глобального split: часть пользователей может появляться только в поздних периодах, что повышает cold-start долю в eval


## 8. Проверка допустимости users/items в eval

In [6]:
train_users = set(train_implicit['userId'].unique())
train_items = set(train_implicit['movieId'].unique())

for_eval = []
for name, df in [('validation', val_implicit), ('test', test_implicit)]:
    new_users = (~df['userId'].isin(train_users)).sum()
    new_items = (~df['movieId'].isin(train_items)).sum()

    warm_df = df[df['userId'].isin(train_users) & df['movieId'].isin(train_items)].copy()
    dropped = len(df) - len(warm_df)

    for_eval.append({
        'split': name,
        'interactions_total': len(df),
        'new_user_interactions': int(new_users),
        'new_item_interactions': int(new_items),
        'warm_interactions': len(warm_df),
        'dropped_for_warm_eval': int(dropped),
        'dropped_pct': round(dropped / max(len(df), 1) * 100, 2),
    })

overlap_summary = pd.DataFrame(for_eval)
display(overlap_summary)

val_implicit_warm = val_implicit[val_implicit['userId'].isin(train_users) & val_implicit['movieId'].isin(train_items)].copy()
test_implicit_warm = test_implicit[test_implicit['userId'].isin(train_users) & test_implicit['movieId'].isin(train_items)].copy()

,split,interactions_total,new_user_interactions,new_item_interactions,warm_interactions,dropped_for_warm_eval,dropped_pct
0,validation,1688735,1456474,93307,196779,1491956,88.35
1,test,1688736,1518976,196044,120890,1567846,92.84



- Для честного сравнения baseline-моделей основной offline eval делаем в **warm-start** режиме
- Полностью новые users/items в val/test фиксируем отдельно как диагностический cold-start слой
- Такой подход не «ломает» метрики моделей, которые не умеют работать с unseen user/item без fallback-механики


## 9. Формирование finished splits для baseline-моделей

In [7]:

full_interactions_explicit = interactions[['userId', 'movieId', 'rating', 'timestamp', 'timestamp_dt']].copy()
train_explicit = full_interactions_explicit[full_interactions_explicit['timestamp'] <= q80].copy()

#сохраняем все таблицы 
final_tables = {
    'full_interactions_explicit': full_interactions_explicit,
    'full_interactions_implicit_raw': raw_implicit,
    'full_interactions_implicit_filtered': filtered_implicit,
    'full_interactions_implicit_capped': capped_implicit,
    'full_interactions_implicit_dropped': dropped_implicit,
    'train_interactions_explicit': train_explicit,
    'train_interactions_implicit': train_implicit,
    'val_interactions_implicit': val_implicit,
    'test_interactions_implicit': test_implicit,
    'val_interactions_implicit_warm': val_implicit_warm,
    'test_interactions_implicit_warm': test_implicit_warm,
}

def save_df_with_fallback(df, path_without_suffix):
    path_without_suffix = Path(path_without_suffix)
    csv_path = path_without_suffix.with_suffix('.csv')
    df.to_csv(csv_path, index=False)
    return csv_path, 'csv'


split_metadata = {
    'active_dataset': active_dataset,
    'min_user_pos': MIN_USER_POS,
    'min_item_pos': MIN_ITEM_POS,
    'user_activity_control_mode': USER_ACTIVITY_CONTROL_MODE,
    'cap_user_interactions': CAP_USER_INTERACTIONS,
    'drop_user_interactions_threshold': DROP_USER_INTERACTIONS_THRESHOLD,
    'dropped_users_count': len(heavy_users_to_drop),
    'main_variant_name': main_variant_name,
    'q80_cutoff_timestamp': q80,
    'q90_cutoff_timestamp': q90,
    'q80_cutoff_datetime': str(pd.to_datetime(q80, unit='s')),
    'q90_cutoff_datetime': str(pd.to_datetime(q90, unit='s')),
}

meta_path = SPLITS_DIR / 'split_metadata.json'
with meta_path.open('w', encoding='utf-8') as f:
    json.dump(split_metadata, f, ensure_ascii=False, indent=2)

final_shapes = pd.DataFrame([
    {
        'dataset': k,
        'rows': len(v),
        'users': v['userId'].nunique() if 'userId' in v.columns else np.nan,
        'items': v['movieId'].nunique() if 'movieId' in v.columns else np.nan,
        'time_min': v['timestamp_dt'].min() if 'timestamp_dt' in v.columns and len(v) > 0 else pd.NaT,
        'time_max': v['timestamp_dt'].max() if 'timestamp_dt' in v.columns and len(v) > 0 else pd.NaT,
    }
    for k, v in final_tables.items()
])
display(final_shapes)



,dataset,rows,users,items,time_min,time_max
0,full_interactions_explicit,33832162,330975,83239,1995-01-09 11:46:44,2023-07-20 08:53:33
1,full_interactions_implicit_raw,16916912,324416,54999,1995-01-09 11:46:44,2023-07-20 08:53:33
2,full_interactions_implicit_filtered,16887355,311102,38716,1995-01-09 11:46:44,2023-07-20 08:53:33
3,full_interactions_implicit_capped,16781981,311102,38681,1995-01-09 11:46:44,2023-07-20 08:53:33
4,full_interactions_implicit_dropped,16478981,310799,37926,1995-01-09 11:46:44,2023-07-20 08:53:33
5,train_interactions_explicit,27075462,280485,49411,1995-01-09 11:46:44,2018-07-11 02:45:50
6,train_interactions_implicit,13509884,264422,29796,1995-01-09 11:46:44,2018-07-11 02:45:48
7,val_interactions_implicit,1688735,30505,28762,2018-07-11 02:46:00,2020-08-22 16:40:07
8,test_interactions_implicit,1688736,30840,28931,2020-08-22 16:40:53,2023-07-20 08:53:33
9,val_interactions_implicit_warm,196779,6638,15772,2018-07-11 02:46:00,2020-08-22 16:28:31


## 10. Подготовка данных для popularity baseline

In [8]:
popularity_stats = (
    train_implicit.groupby('movieId')
    .agg(
        positive_interactions=('movieId', 'size'),
        unique_users=('userId', 'nunique')
    )
    .sort_values(['positive_interactions', 'unique_users'], ascending=False)
    .reset_index()
)
popularity_stats['popularity_rank'] = np.arange(1, len(popularity_stats) + 1)
pop_path, pop_fmt = save_df_with_fallback(popularity_stats, PROCESSED_DIR / 'train_item_popularity')

train_user_history = (
    train_implicit.groupby('userId')['movieId']
    .agg(list)
    .reset_index(name='seen_movie_ids')
)

# сохраняем в long-форме для удобства последующей обработки
train_user_history_long = train_implicit[['userId', 'movieId']].drop_duplicates().copy()
history_path, hist_fmt = save_df_with_fallback(train_user_history_long, PROCESSED_DIR / 'train_user_history_long')

candidate_items = pd.DataFrame({'movieId': sorted(train_implicit['movieId'].unique())})
cand_path, cand_fmt = save_df_with_fallback(candidate_items, PROCESSED_DIR / 'candidate_items_train')

display(popularity_stats.head(10))


,movieId,positive_interactions,unique_users,popularity_rank
0,318,83235,83235,1
1,296,70705,70705,2
2,356,69327,69327,3
3,593,67901,67901,4
4,2571,63237,63237,5
5,260,61151,61151,6
6,527,57113,57113,7
7,2959,51222,51222,8
8,50,50790,50790,9
9,1196,49803,49803,10


- Для popularity baseline достаточно train implicit interactions и агрегированной item-popularity таблицы
- `train_user_history_long` нужен, чтобы исключать уже просмотренные/оценённые объекты при генерации рекомендаций


## 11. Подготовка данных для item-item CF

In [9]:
item_item_source = train_implicit[['userId', 'movieId']].drop_duplicates().copy()

user_map = pd.DataFrame({'userId': sorted(item_item_source['userId'].unique())})
user_map['user_idx'] = np.arange(len(user_map), dtype=np.int64)

item_map = pd.DataFrame({'movieId': sorted(item_item_source['movieId'].unique())})
item_map['item_idx'] = np.arange(len(item_map), dtype=np.int64)

indexed = item_item_source.merge(user_map, on='userId', how='left').merge(item_map, on='movieId', how='left')
indexed['value'] = 1.0

u_path, u_fmt = save_df_with_fallback(user_map, PROCESSED_DIR / 'user_mapping')
i_path, i_fmt = save_df_with_fallback(item_map, PROCESSED_DIR / 'item_mapping')
ix_path, ix_fmt = save_df_with_fallback(indexed[['userId', 'movieId', 'user_idx', 'item_idx', 'value']], PROCESSED_DIR / 'train_interactions_indexed')

display(indexed.head())


,userId,movieId,user_idx,item_idx,value
0,1,1,0,0,1.0
1,1,110,0,108,1.0
2,1,158,0,156,1.0
3,1,260,0,257,1.0
4,1,356,0,351,1.0


- `item-item CF` строится на user-item взаимодействиях train-периода
- Индексированные таблицы позволяют в следующем ноутбуке быстро собрать sparse-матрицу и считать similarity


## 12. Подготовка данных для matrix factorization

In [10]:
# Для MF используем тот же train implicit, но сохраняем и explicit train
mf_train_implicit = train_implicit[['userId', 'movieId', 'timestamp', 'timestamp_dt']].copy()
mf_train_implicit['target'] = 1.0

mf_train_explicit = train_explicit[['userId', 'movieId', 'rating', 'timestamp', 'timestamp_dt']].copy()

mf_implicit_path, mf_implicit_fmt = save_df_with_fallback(mf_train_implicit, PROCESSED_DIR / 'mf_train_implicit')
mf_explicit_path, mf_explicit_fmt = save_df_with_fallback(mf_train_explicit, PROCESSED_DIR / 'mf_train_explicit_reference')

display(mf_train_implicit.head())


,userId,movieId,timestamp,timestamp_dt,target
0,1,1,1225734739,2008-11-03 17:52:19,1.0
1,1,110,1225865086,2008-11-05 06:04:46,1.0
2,1,158,1225733503,2008-11-03 17:31:43,1.0
3,1,260,1225735204,2008-11-03 18:00:04,1.0
4,1,356,1225735119,2008-11-03 17:58:39,1.0


- Вход для MF концептуально похож на item-item: нужна user-item матрица train-периода
- Отличие: factorization учит латентные факторы пользователей и объектов, поэтому mapping и консистентный train-split критичны


## 13. Подготовка metadata для hybrid

In [11]:
item_meta = movies.copy()
keep_cols = [c for c in ['movieId', 'title', 'genres'] if c in item_meta.columns]
item_meta = item_meta[keep_cols].copy()

item_meta['release_year'] = pd.to_numeric(
    item_meta['title'].astype(str).str.extract(r'\((\d{4})\)$')[0],
    errors='coerce'
)
item_meta['genres'] = item_meta['genres'].fillna('(no genres listed)').astype(str)
item_meta['genres_list'] = item_meta['genres'].str.split('|')
item_meta['n_genres'] = item_meta['genres_list'].apply(len)

meta_path, meta_fmt = save_df_with_fallback(item_meta, PROCESSED_DIR / 'item_metadata_cleaned')
display(item_meta.head())


tags_clean = tags[['movieId', 'tag']].dropna().copy()
tags_clean['tag_clean'] = tags_clean['tag'].astype(str).str.strip().str.lower()
tags_clean = tags_clean[tags_clean['tag_clean'] != '']

tag_stats = tags_clean.groupby('movieId').agg(
    n_tags=('tag_clean', 'size'),
    n_unique_tags=('tag_clean', 'nunique')
).reset_index()

tag_path, tag_fmt = save_df_with_fallback(tag_stats, PROCESSED_DIR / 'item_tag_stats')
display(tag_stats.head())


,movieId,title,genres,release_year,genres_list,n_genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,"[Adventure, Animation, Children, Comedy, Fantasy]",5
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,"[Adventure, Children, Fantasy]",3
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,"[Comedy, Romance]",2
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,"[Comedy, Drama, Romance]",3
4,5,Father of the Bride Part II (1995),Comedy,1995.0,[Comedy],1


,movieId,n_tags,n_unique_tags
0,1,1440,449
1,2,653,356
2,3,36,30
3,4,13,10
4,5,68,36


## 14. Подготовка offline evaluation protocol

In [12]:
# Ground truth: positive interactions в соответствующем split
val_ground_truth = val_implicit_warm[['userId', 'movieId']].drop_duplicates().copy()
test_ground_truth = test_implicit_warm[['userId', 'movieId']].drop_duplicates().copy()

train_history = train_implicit[['userId', 'movieId']].drop_duplicates().copy()
seen_items_by_user = train_history.copy()

eval_users_val = pd.DataFrame({'userId': sorted(val_ground_truth['userId'].unique())})
eval_users_test = pd.DataFrame({'userId': sorted(test_ground_truth['userId'].unique())})

# Сохранение eval-ready артефактов
art_eval = {
    'train_user_history': train_history,
    'val_ground_truth': val_ground_truth,
    'test_ground_truth': test_ground_truth,
    'seen_items_by_user': seen_items_by_user,
    'eval_users_val': eval_users_val,
    'eval_users_test': eval_users_test,
}

for name, df in art_eval.items():
    pth, fmt = save_df_with_fallback(df, EVAL_DIR / name)

print('Saved eval artifacts:', ', '.join(art_eval.keys()))



Saved eval artifacts: train_user_history, val_ground_truth, test_ground_truth, seen_items_by_user, eval_users_val, eval_users_test


### Evaluation Protocol 

`ground_truth`: positive interactions (`rating >= 4`) в warm validation/test

`eval_users`: пользователи из warm validation/test

`candidate_space`: только items из train

`seen-item filter`: исключаем items, уже виденные пользователем в train

`metrics`: Recall@K, Precision@K, MAP@K, NDCG@K, Coverage, Catalog Coverage

### Объяснение метрик

`Recall@K` показывает долю релевантных объектов, найденных в топ-K

`Precision@K` показывает точность рекомендаций в топ-K

`MAP@K` показывает качество ранжирования с учетом позиций релевантных объектов

`NDCG@K` показывает качество порядка рекомендаций с позиционным дисконтированием

`Coverage` показывает, какая доля объектов вообще рекомендуется

`Catalog Coverage` показывает, насколько система использует каталог, а не только head-объекты

### Что это значит для рекомендательной системы

- Этот ноутбук не считает качество моделей, но задаёт воспроизводимый и честный **evaluation protocol**
- Все baseline-модели будут сравниваться на одинаковых warm-start ground truth наборах и в едином candidate space
- Это критично для корректного сравнения моделей

## 16. Итоговые выводы и следующий шаг

### Зафиксированные решения
1. Основная постановка: **implicit recommendation**, positive = `rating >= 4`
2. Split: **chronological** (`train=oldest 80%`, `val=next 10%`, `test=newest 10%`)
3. Offline eval для baseline-моделей: **warm-start subset** в validation/test

### Подготовленные артефакты
- processed interactions (explicit + implicit)
- split datasets (train/val/test + warm eval)
- popularity inputs (item popularity, candidate space, user history)
- index mappings и indexed interactions для item-item/MF
- eval protocol артефакты (ground truth, eval users, seen items)
- metadata-таблицы для будущего hybrid этапа (если исходные файлы доступны)
